In [ ]:
!pip install -q ultralytics open-clip-torch hnswlib tqdm pandas transformers accelerate

In [ ]:
# ============================================================
#  NOTEBOOK C — Fine-tuned CLIP + Frozen BLIP-2
#  Seeds: 039,003,113,528 | Alphas: 0.7,0.5
#  Metrics: HR@K, Recall@K, mAP@K, NDCG@K (K∈{5,10,15})
# ============================================================
# !pip install -q ultralytics open-clip-torch hnswlib tqdm pandas transformers accelerate

import os, json, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import hnswlib
import open_clip
from ultralytics import YOLO
from torchvision import transforms
from transformers import BlipProcessor, BlipForImageTextRetrieval, Blip2Processor, Blip2ForConditionalGeneration

NOTEBOOK_START = time.time()


CFG = dict(
    img_root="/kaggle/input/datasets/fireworksbads/project2/project2/img/img",
    eval_file="/kaggle/input/datasets/fireworksbads/project2/project2/eval/list_eval_partition.txt",
    yolo_weights="/kaggle/input/datasets/fireworksbads/output/detect/train/weights/best.pt",
    clip_model="ViT-B-32", clip_pretrain="openai", image_size=224,
    seeds=[39,3,113,528], alphas=[0.7],
    finetune_epochs=15, finetune_lr=1e-5, finetune_batch=32,
    temperature=0.1, unfreeze_last_n=4,
    work="/kaggle/working/",
    hnsw_M=32, hnsw_efConstruct=200, hnsw_efSearch=50,
    yolo_batch=64, yolo_pad=10, clip_batch=128,
    TIME_BUDGET_SECS=int(11.5*3600),
    top_k=[5,10,15],
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [ ]:

def _elapsed_h(): return (time.time()-NOTEBOOK_START)/3600
def _time_left(): return CFG["TIME_BUDGET_SECS"]-(time.time()-NOTEBOOK_START)
def _check_budget(label=""):
    if _time_left()<600:
        print(f"\n⚠ Budget exhausted at [{label}]."); raise SystemExit("Budget exceeded")

print(f"Device: {CFG['device']}  Budget: {CFG['TIME_BUDGET_SECS']/3600:.1f}h")

# ── Splits ────────────────────────────────────────────────────
split_df = pd.read_csv(CFG["eval_file"],sep=r"\s+",header=0,
    names=["image_name","item_id","evaluation_status"],skiprows=1)
split_df = split_df.applymap(lambda x: x.strip() if isinstance(x,str) else x)

train_df   = split_df[split_df["evaluation_status"]=="train"].reset_index(drop=True)
gallery_df = split_df[split_df["evaluation_status"]=="gallery"].reset_index(drop=True)
query_df   = split_df[split_df["evaluation_status"]=="query"].reset_index(drop=True)

train_paths=  [os.path.join(CFG["img_root"],p) for p in train_df["image_name"]]
gallery_paths=[os.path.join(CFG["img_root"],p) for p in gallery_df["image_name"]]
gallery_items=gallery_df["item_id"].tolist()
gallery_item_counts={}
for it in gallery_items: gallery_item_counts[it]=gallery_item_counts.get(it,0)+1
query_paths=[os.path.join(CFG["img_root"],p) for p in query_df["image_name"]]
query_items=query_df["item_id"].tolist()
print(f"Train:{len(train_df)} Gallery:{len(gallery_df)} Query:{len(query_df)}")


In [ ]:

# ── YOLO ──────────────────────────────────────────────────────
model_yolo = YOLO(CFG["yolo_weights"]); model_yolo.to(CFG["device"])
print("YOLO loaded.")

def _crop_path(p):
    return os.path.join(CFG["work"],p.replace("/","_").replace("\\","_").lstrip("_")+".jpg")

def yolo_crop_batch(img_paths):
    pad=CFG["yolo_pad"]
    to_process=[p for p in img_paths if not os.path.exists(_crop_path(p))]
    if to_process:
        results=model_yolo.predict(to_process,device=CFG["device"],verbose=False)
        for img_path,result in zip(to_process,results):
            out=_crop_path(img_path); img=Image.open(img_path).convert("RGB")
            boxes=result.boxes
            if boxes is not None and len(boxes)>0:
                areas=(boxes.xyxy[:,2]-boxes.xyxy[:,0])*(boxes.xyxy[:,3]-boxes.xyxy[:,1])
                best=boxes[areas.argmax()]
                x1,y1,x2,y2=map(int,best.xyxy[0].tolist()); W,H=img.size
                img.crop((max(0,x1-pad),max(0,y1-pad),min(W,x2+pad),min(H,y2+pad))).save(out)
            else: img.save(out)
    return [_crop_path(p) for p in img_paths]


In [ ]:

# ── BLIP-2 Captioning (same approach as B) ────────────────────
BS=CFG["yolo_batch"]
CATALOG=os.path.join(CFG["work"],"catalog.json")

print(f"\n[{_elapsed_h():.2f}h] Loading BLIP-2 …")
blip_proc=Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model=Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",torch_dtype=torch.float16,device_map="auto").eval()
for p in blip_model.parameters(): p.requires_grad=False
print("BLIP-2 loaded (frozen, float16).")

CPROMPT='Question: Describe this clothing item including color, type, and style. Answer:'
DEVICE=CFG["device"]

def caption_batch_fast(img_paths):
    captions=[]
    for i in range(0,len(img_paths),BS):
        batch=img_paths[i:i+BS]
        images=[Image.open(p).convert("RGB") for p in batch]
        prompts=[CPROMPT]*len(images)
        inp=blip_proc(images=images,text=prompts,return_tensors='pt',padding=True).to(DEVICE,torch.float16)
        with torch.no_grad():
            out=blip_model.generate(**inp,max_new_tokens=20,do_sample=False,num_beams=1)
        texts=blip_proc.batch_decode(out,skip_special_tokens=True)
        texts=[t.split('Answer:')[-1].strip() if 'Answer:' in t else t.strip() for t in texts]
        captions.extend(texts)
    return captions

# Build catalog (crop + caption) with checkpointing
if os.path.exists(CATALOG):
    with open(CATALOG) as f: catalog=json.load(f)
    done={r["image_path"] for r in catalog}
    remaining=[p for p in gallery_paths if p not in done]
    print(f"[{_elapsed_h():.2f}h] Catalog loaded: {len(catalog)} done, {len(remaining)} remaining.")
else:
    catalog,remaining=[],gallery_paths

for i in tqdm(range(0,len(remaining),BS),desc="Catalog (crop+caption)"):
    _check_budget("catalog loop")
    batch=remaining[i:i+BS]
    try:
        crops=yolo_crop_batch(batch)
        captions=caption_batch_fast(crops)
        for orig,crop,cap in zip(batch,crops,captions):
            item_id=orig.replace("\\","/").split("/")[-2]
            catalog.append({"item_id":item_id,"image_path":orig,"cropped_path":crop,"caption":cap})
    except Exception as e:
        print(f"  [skip batch {i}]: {e}")
    if (i//BS)%50==0 and i>0:
        with open(CATALOG,"w") as f: json.dump(catalog,f)

with open(CATALOG,"w") as f: json.dump(catalog,f,indent=2)
print(f"[{_elapsed_h():.2f}h] Catalog done — {len(catalog)} records")

# Offload BLIP-2 to free VRAM for CLIP fine-tuning
del blip_model,blip_proc
torch.cuda.empty_cache()
print("BLIP-2 unloaded — VRAM freed.")

# Build gallery data from catalog
gallery_crops=[r["cropped_path"] for r in catalog]
gallery_captions=[r["caption"] for r in catalog]
gallery_items_cat=[r["item_id"] for r in catalog]
gallery_meta=[{"item_id":r["item_id"],"image_path":r["image_path"],
    "cropped_path":r["cropped_path"],"caption":r["caption"]} for r in catalog]

In [ ]:

# ── Crop train ────────────────────────────────────────────────
train_crop_json=os.path.join(CFG["work"],"train_crops.json")
if os.path.exists(train_crop_json):
    with open(train_crop_json) as f: train_records=json.load(f)
else:
    train_records=[]
    for i in tqdm(range(0,len(train_paths),BS),desc="YOLO crop train"):
        batch=train_paths[i:i+BS]
        try:
            crops=yolo_crop_batch(batch)
            for orig,crop in zip(batch,crops):
                train_records.append({"item_id":orig.replace("\\","/").split("/")[-2],"cropped_path":crop})
        except: pass
    with open(train_crop_json,"w") as f: json.dump(train_records,f)

# ── Fine-tune dataset (PairBatchSampler from pipeline_part1) ─
train_tf=transforms.Compose([
    transforms.RandomResizedCrop(CFG["image_size"],scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466,0.4578275,0.40821073),(0.26862954,0.26130258,0.27577711)),
])
eval_tf=transforms.Compose([
    transforms.Resize((CFG["image_size"],CFG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466,0.4578275,0.40821073),(0.26862954,0.26130258,0.27577711)),
])



In [ ]:

class FashionDS(Dataset):
    def __init__(self,records,tr):
        self.df=pd.DataFrame(records); self.tr=tr
        self.id2int={v:i for i,v in enumerate(self.df['item_id'].unique())}
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        try: img=Image.open(row['cropped_path']).convert('RGB')
        except: img=Image.new('RGB',(224,224))
        return self.tr(img), self.id2int[row['item_id']]

class PairBatchSampler(Sampler):
    def __init__(self,ds,bs):
        self.bs=bs; self.pn=bs//2; self.id2idx=defaultdict(list)
        for i in range(len(ds)): self.id2idx[ds.df.iloc[i]['item_id']].append(i)
        self.valid=[k for k,v in self.id2idx.items() if len(v)>=2]
    def __iter__(self):
        ids=list(self.valid); random.shuffle(ids); batch=[]
        for iid in ids:
            batch.extend(random.sample(self.id2idx[iid],2))
            if len(batch)>=self.bs: yield batch[:self.bs]; batch=batch[self.bs:]
    def __len__(self): return len(self.valid)//self.pn
def info_nce(embs, labels, temp=CFG["temperature"]):
    """InfoNCE with positive-pair mask. Numerically stabilized with fp32 + clamp."""
    embs = F.normalize(embs.float(), dim=1)
    labels = labels.to(embs.device)
    sim = embs @ embs.T / temp
    sim = sim.clamp(-50, 50)  # prevent overflow in log_softmax
    # positive mask: same label = 1, diagonal = 0
    mask_pos = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
    mask_pos.fill_diagonal_(0)
    # mask out self-similarity
    mask_self = torch.eye(sim.size(0), device=sim.device).bool()
    sim = sim.masked_fill(mask_self, -1e9)  # large negative instead of -inf
    # log softmax
    log_prob = F.log_softmax(sim, dim=1)
    # mean of log-probs at positive positions
    n_pos = mask_pos.sum(1).clamp(min=1)
    loss = -(log_prob * mask_pos).sum(1) / n_pos
    return loss.mean()

In [ ]:
import os
if os.path.exists('/kaggle/working/clip_best.pt'):
    os.remove('/kaggle/working/clip_best.pt')
    print("Deleted old checkpoint")


In [ ]:
FINETUNE_SEED=CFG["seeds"][0]
best_ckpt=os.path.join(CFG["work"],"clip_best.pt")

if not os.path.exists(best_ckpt):
    random.seed(FINETUNE_SEED); np.random.seed(FINETUNE_SEED); torch.manual_seed(FINETUNE_SEED)
    clip_ft,_,_=open_clip.create_model_and_transforms(CFG["clip_model"],pretrained=CFG["clip_pretrain"])
    clip_ft=clip_ft.float().to(CFG["device"])  # explicit fp32
    for p in clip_ft.parameters(): p.requires_grad=False
    for blk in list(clip_ft.visual.transformer.resblocks)[-CFG["unfreeze_last_n"]:]:
        for p in blk.parameters(): p.requires_grad=True
    if hasattr(clip_ft.visual,"proj") and clip_ft.visual.proj is not None:
        clip_ft.visual.proj.requires_grad=False
    for p in clip_ft.visual.ln_post.parameters(): p.requires_grad=False
    for p in clip_ft.transformer.parameters(): p.requires_grad=False
    print(f"Trainable: {sum(p.numel() for p in clip_ft.parameters() if p.requires_grad):,}")

    ds=FashionDS(train_records,train_tf)
    samp=PairBatchSampler(ds,CFG["finetune_batch"])
    loader=DataLoader(ds,batch_sampler=samp,num_workers=2,pin_memory=True)
    opt=torch.optim.AdamW(filter(lambda p:p.requires_grad,clip_ft.parameters()),lr=CFG["finetune_lr"])
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CFG["finetune_epochs"])
    best_loss=float('inf')

    for ep in range(1,CFG["finetune_epochs"]+1):
        _check_budget(f"ft ep {ep}"); clip_ft.train(); tot=0
        for imgs,lbs in tqdm(loader,desc=f"Ep {ep}/{CFG['finetune_epochs']}",leave=False):
            imgs=imgs.to(CFG["device"])
            embs=clip_ft.encode_image(imgs)
            loss=info_nce(embs,lbs)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(clip_ft.parameters(),max_norm=1.0)
            opt.step()
            tot+=loss.item()
        sched.step(); avg=tot/len(loader)
        print(f"  Ep {ep} loss={avg:.4f}")
        if avg<best_loss: best_loss=avg; torch.save({"epoch":ep,"loss":avg,"model_state":clip_ft.state_dict()},best_ckpt)
    del clip_ft; torch.cuda.empty_cache()
else:
    print(f"Checkpoint found: {best_ckpt}")


In [ ]:

# ── Load fine-tuned CLIP ──────────────────────────────────────
clip_ft,_,_=open_clip.create_model_and_transforms(CFG["clip_model"],pretrained=CFG["clip_pretrain"])
ckpt_data=torch.load(best_ckpt,map_location="cpu")
clip_ft.load_state_dict(ckpt_data["model_state"])
clip_ft=clip_ft.to(CFG["device"]).eval()
tokenizer=open_clip.get_tokenizer(CFG["clip_model"])
print(f"Fine-tuned CLIP loaded (ep {ckpt_data['epoch']}, loss={ckpt_data['loss']:.4f})")

# ── Cache embeddings ──────────────────────────────────────────
@torch.no_grad()
def encode_images_batched(model,img_paths):
    vecs=[]
    for i in tqdm(range(0,len(img_paths),CFG["clip_batch"]),desc="  img enc",leave=False):
        batch=img_paths[i:i+CFG["clip_batch"]]
        ts=[eval_tf(Image.open(p).convert("RGB")) if os.path.exists(p)
            else torch.zeros(3,CFG["image_size"],CFG["image_size"]) for p in batch]
        tb=torch.stack(ts).to(CFG["device"])
        with torch.amp.autocast(CFG["device"],enabled=CFG["device"]!="cpu"):
            feat=model.encode_image(tb)
        vecs.append(F.normalize(feat,dim=-1).cpu().float().numpy())
    return np.concatenate(vecs)

@torch.no_grad()
def encode_texts_batched(model,captions):
    vecs=[]
    for i in tqdm(range(0,len(captions),CFG["clip_batch"]),desc="  txt enc",leave=False):
        batch=captions[i:i+CFG["clip_batch"]]
        tok=tokenizer(batch).to(CFG["device"])
        with torch.amp.autocast(CFG["device"],enabled=CFG["device"]!="cpu"):
            feat=model.encode_text(tok)
        vecs.append(F.normalize(feat,dim=-1).cpu().float().numpy())
    return np.concatenate(vecs)


In [ ]:

img_cache=os.path.join(CFG["work"],"gal_img_ft.npy")
txt_cache=os.path.join(CFG["work"],"gal_txt_ft.npy")
q_cache  =os.path.join(CFG["work"],"q_img_ft.npy")

if os.path.exists(img_cache): gal_img=np.load(img_cache)
else: gal_img=encode_images_batched(clip_ft,gallery_crops); np.save(img_cache,gal_img)

if os.path.exists(txt_cache): gal_txt=np.load(txt_cache)
else: gal_txt=encode_texts_batched(clip_ft,gallery_captions); np.save(txt_cache,gal_txt)

query_crops=[]
for i in tqdm(range(0,len(query_paths),BS),desc="YOLO crop queries"):
    query_crops.extend(yolo_crop_batch(query_paths[i:i+BS]))

if os.path.exists(q_cache): q_img=np.load(q_cache)
else: q_img=encode_images_batched(clip_ft,query_crops); np.save(q_cache,q_img)

# ── Metrics (TWO recall formulas) ─────────────────────────────
def hit_rate_at_k(ret,rel,k): return int(len(set(ret[:k])&rel)>0)
def recall_at_k(ret,rel,k,nr): return sum(1 for r in ret[:k] if r in rel)/max(1,nr)
def ap_at_k(ret,rel,k,nr):
    h,s=0,0.0
    for rk,r in enumerate(ret[:k],1):
        if r in rel: h+=1; s+=h/rk
    return s/max(1,min(k,nr))
def ndcg_at_k(ret,rel,k,nr):
    d=sum(1/np.log2(i+2) for i,r in enumerate(ret[:k]) if r in rel)
    ideal=sum(1/np.log2(i+2) for i in range(min(k,nr)))
    return d/ideal if ideal>0 else 0.0

def compute_metrics(all_ret,all_rel,all_nr,top_k):
    out={}
    for k in top_k:
        out[k]={
            "HR":    (float(np.mean([hit_rate_at_k(r,rel,k) for r,rel in zip(all_ret,all_rel)])),
                      float(np.std ([hit_rate_at_k(r,rel,k) for r,rel in zip(all_ret,all_rel)]))),
            "Recall":(float(np.mean([recall_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)])),
                      float(np.std ([recall_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)]))),
            "mAP":   (float(np.mean([ap_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)])),
                      float(np.std ([ap_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)]))),
            "NDCG":  (float(np.mean([ndcg_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)])),
                      float(np.std ([ndcg_at_k(r,rel,k,nr) for r,rel,nr in zip(all_ret,all_rel,all_nr)]))),
        }
    return out


In [ ]:

# ── BLIP ITM Re-ranker ────────────────────────────────────────
print("\nLoading BLIP ITM...")
itm_proc=BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")
itm_mdl=BlipForImageTextRetrieval.from_pretrained(
    "Salesforce/blip-itm-base-coco",torch_dtype=torch.float16).to(CFG["device"]).eval()

def itm_rerank(qcrop_path,cands,meta,caps):
    if not caps: return cands
    valid_caps,valid_indices=[],[]
    for i,c_idx in enumerate(cands):
        cap=meta[c_idx].get("caption","")
        if cap: valid_caps.append(cap); valid_indices.append(i)
    if not valid_caps: return cands
    qimg=Image.open(qcrop_path).convert("RGB"); scores=[]
    for i in range(0,len(valid_caps),32):
        vc=valid_caps[i:i+32]
        try:
            inp=itm_proc(images=[qimg]*len(vc),text=vc,return_tensors="pt",padding=True).to(CFG["device"])
            inp["pixel_values"]=inp["pixel_values"].half()
            with torch.no_grad():
                out=itm_mdl(**inp,use_itm_head=True)
                scores.extend(F.softmax(out.itm_score,dim=1)[:,1].cpu().numpy().tolist())
        except: scores.extend([0.0]*len(vc))
    fs=np.zeros(len(cands))
    for s,idx in zip(scores,valid_indices): fs[idx]=s
    return [cands[i] for i in np.argsort(-fs,kind='stable')]

# ── HNSW helper ───────────────────────────────────────────────
def build_hnsw(vectors):
    idx=hnswlib.Index(space='cosine',dim=vectors.shape[1])
    idx.init_index(max_elements=len(vectors),ef_construction=CFG["hnsw_efConstruct"],M=CFG["hnsw_M"])
    idx.add_items(vectors,np.arange(len(vectors))); idx.set_ef(CFG["hnsw_efSearch"])
    return idx

In [ ]:



# ── Main loop — 4 seeds × 2 alphas ───────────────────────────
all_results_C={}; max_k=max(CFG["top_k"])
caption_lookup=True  # signal to itm_rerank that captions exist in meta

for seed in CFG["seeds"]:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    print(f"\n{'='*62}\n  SEED {seed:03d}   [{_elapsed_h():.2f}h | {_time_left()/3600:.1f}h left]\n{'='*62}")
    _check_budget(f"seed {seed}")
    perm=np.random.permutation(len(query_paths))
    q_paths_s=[query_paths[i] for i in perm]
    q_crops_s=[query_crops[i] for i in perm]
    q_items_s=[query_items[i] for i in perm]
    q_embs_s=q_img[perm]

    for alpha in CFG["alphas"]:
        _check_budget(f"seed {seed} a={alpha}")
        run_key=f"seed{seed:03d}_alpha{int(alpha*100):03d}"
        print(f"\n  ── {run_key} ──")
        fused=alpha*gal_img+(1.0-alpha)*gal_txt
        norms=np.linalg.norm(fused,axis=1,keepdims=True).clip(min=1e-8)
        fused_n=(fused/norms).astype("float32")
        index=build_hnsw(fused_n)
        index.save_index(os.path.join(CFG["work"],f"index_C_{run_key}.bin"))

        all_ret,all_rel=[],[]
        for qi in tqdm(range(len(q_paths_s)),desc="  Search & ITM",leave=False):
            qvec=q_embs_s[qi].reshape(1,-1).astype("float32")
            labels,_=index.knn_query(qvec,k=max_k)
            cands=list(labels[0])
            reranked=itm_rerank(q_crops_s[qi],cands,gallery_meta,caption_lookup)
            all_ret.append([gallery_meta[i]["item_id"] for i in reranked])
            all_rel.append({q_items_s[qi]})

        all_nr=[gallery_item_counts.get(list(rel)[0],1) for rel in all_rel]
        metrics=compute_metrics(all_ret,all_rel,all_nr,CFG["top_k"])
        all_results_C[run_key]=metrics
        for k in CFG["top_k"]:
            hr,hrs=metrics[k]["HR"]; r,rs=metrics[k]["Recall"]
            m,ms=metrics[k]["mAP"]; n,ns=metrics[k]["NDCG"]
            print(f"    K={k:2d} HR={hr:.4f}±{hrs:.4f} R={r:.4f}±{rs:.4f} mAP={m:.4f}±{ms:.4f} NDCG={n:.4f}±{ns:.4f}")
        del index,fused,fused_n



In [ ]:
# ── Aggregate ─────────────────────────────────────────────────
print(f"\n\n{'='*80}\n  CONDITION C — Aggregated\n{'='*80}")
print(f"{'Config':<28} {'K':>3}  {'HR@K':>12}  {'Recall@K':>12}  {'mAP':>12}  {'NDCG':>12}")
print("-"*84)

aggregated_C={}
for alpha in CFG["alphas"]:
    ak=f"alpha{int(alpha*100):03d}"
    sms=[all_results_C[f"seed{s:03d}_{ak}"] for s in CFG["seeds"] if f"seed{s:03d}_{ak}" in all_results_C]
    if not sms: continue
    agg={}
    for k in CFG["top_k"]:
        agg[k]={met:(float(np.mean([sm[k][met][0] for sm in sms])),
                      float(np.std([sm[k][met][0] for sm in sms]))) for met in ["HR","Recall","mAP","NDCG"]}
    aggregated_C[ak]=agg
    for k in CFG["top_k"]:
        hr,hrs=agg[k]["HR"]; r,rs=agg[k]["Recall"]; m,ms=agg[k]["mAP"]; n,ns=agg[k]["NDCG"]
        print(f"  C α={alpha}       {k:>3}  {hr:.4f}±{hrs:.4f}  {r:.4f}±{rs:.4f}  {m:.4f}±{ms:.4f}  {n:.4f}±{ns:.4f}")

# ── Save ──────────────────────────────────────────────────────
out_path=os.path.join(CFG["work"],"results_C.json")
with open(out_path,"w") as f:
    json.dump({"seeds":CFG["seeds"],"alphas":CFG["alphas"],"finetune_ckpt":best_ckpt,
        "per_seed":{k:{str(kk):{m:list(v) for m,v in vv.items()} for kk,vv in vs.items()}
            for k,vs in all_results_C.items()},
        "aggregated":{k:{str(kk):{m:list(v) for m,v in vv.items()} for kk,vv in vs.items()}
            for k,vs in aggregated_C.items()},
    },f,indent=2)
print(f"\n[{_elapsed_h():.2f}h] Results → {out_path}")
print("✅ Notebook C complete.")